In [1]:
import sqlite3
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report,
    accuracy_score
)

from xgboost import XGBClassifier

DB_PATH = "../data/soccer.db"

conn = sqlite3.connect(DB_PATH)

In [2]:
TRAIN_MAX_SEASON = 2023
TEST_MIN_SEASON = 2024

EUROPEAN_LEAGUES = [
    1,   # Premier League
    2,   # La Liga
    3,   # Serie A
    4,   # Bundesliga
    5    # Ligue 1
]

COMPETITION_ID_TO_NAME = {
    1: "Premier League",
    2: "La Liga",
    3: "Serie A",
    4: "Bundesliga",
    5: "Ligue 1"
}

COMPETITION_NAME_TO_ID = {
    name: cid for cid, name in COMPETITION_ID_TO_NAME.items()
}

In [3]:
matches_query = """
SELECT
    *
FROM matches
"""

matches_df = pd.read_sql(
    matches_query,
    conn
)

matches_df = matches_df[
    matches_df["competition_id"].isin(EUROPEAN_LEAGUES)
].copy()

context_query = """
SELECT
    *
FROM match_context
"""

context_df = pd.read_sql(
    context_query,
    conn
)

context_df = context_df.drop(columns=["id"])

matches_features = matches_df.merge(
    context_df,
    left_on="id",
    right_on="match_id",
    how="left"
)

matches_features = matches_features.sort_values(
    by="date"
).reset_index(drop=True)

In [4]:
matches_features["home_elo"] = 1500.0
matches_features["away_elo"] = 1500.0

elo_ratings = {}

K_FACTOR = 20

for idx, row in matches_features.iterrows():

    home_team = row["home_team_id"]
    away_team = row["away_team_id"]

    home_elo = elo_ratings.get(home_team, 1500)
    away_elo = elo_ratings.get(away_team, 1500)

    matches_features.at[idx, "home_elo"] = home_elo
    matches_features.at[idx, "away_elo"] = away_elo

    expected_home = (
        1 /
        (
            1 +
            10 ** (
                (away_elo - home_elo) / 400
            )
        )
    )

    expected_away = 1 - expected_home

    if row["home_goals"] > row["away_goals"]:

        actual_home = 1
        actual_away = 0

    elif row["home_goals"] < row["away_goals"]:

        actual_home = 0
        actual_away = 1

    else:

        actual_home = 0.5
        actual_away = 0.5

    new_home_elo = (
        home_elo +
        K_FACTOR *
        (
            actual_home -
            expected_home
        )
    )

    new_away_elo = (
        away_elo +
        K_FACTOR *
        (
            actual_away -
            expected_away
        )
    )

    elo_ratings[home_team] = new_home_elo
    elo_ratings[away_team] = new_away_elo

In [5]:
matches_features["home_win"] = (
    matches_features["home_goals"] >
    matches_features["away_goals"]
)

matches_features["away_win"] = (
    matches_features["away_goals"] >
    matches_features["home_goals"]
)

matches_features["is_draw"] = (
    matches_features["home_goals"] ==
    matches_features["away_goals"]
)

matches_features["home_dnb"] = (
    matches_features["home_goals"] >
    matches_features["away_goals"]
)

matches_features["away_dnb"] = (
    matches_features["away_goals"] >
    matches_features["home_goals"]
)

matches_features["match_result_3class"] = np.select(
    [
        matches_features["home_win"],
        matches_features["is_draw"],
        matches_features["away_win"]
    ],
    [
        0,
        1,
        2
    ]
)

matches_features["round_number"] = (
    matches_features["round"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
    .astype(float)
)

In [6]:
matches_features["home_draw_rate_last_10"] = (
    matches_features
    .groupby("home_team_id")["is_draw"]
    .transform(
        lambda x:
        x.shift(1).rolling(10, min_periods=1).mean()
    )
)

matches_features["away_draw_rate_last_10"] = (
    matches_features
    .groupby("away_team_id")["is_draw"]
    .transform(
        lambda x:
        x.shift(1).rolling(10, min_periods=1).mean()
    )
)

matches_features["home_rolling_scored"] = (
    matches_features
    .groupby("home_team_id")["home_goals"]
    .transform(
        lambda x:
        x.shift(1).rolling(5, min_periods=1).mean()
    )
)

matches_features["home_rolling_conceded"] = (
    matches_features
    .groupby("home_team_id")["away_goals"]
    .transform(
        lambda x:
        x.shift(1).rolling(5, min_periods=1).mean()
    )
)

matches_features["away_rolling_scored"] = (
    matches_features
    .groupby("away_team_id")["away_goals"]
    .transform(
        lambda x:
        x.shift(1).rolling(5, min_periods=1).mean()
    )
)

matches_features["away_rolling_conceded"] = (
    matches_features
    .groupby("away_team_id")["home_goals"]
    .transform(
        lambda x:
        x.shift(1).rolling(5, min_periods=1).mean()
    )
)

In [7]:
matches_features["elo_diff"] = (
    matches_features["home_elo"] -
    matches_features["away_elo"]
).abs()

matches_features["attack_diff"] = (
    matches_features["home_rolling_scored"] -
    matches_features["away_rolling_scored"]
).abs()

matches_features["defense_diff"] = (
    matches_features["home_rolling_conceded"] -
    matches_features["away_rolling_conceded"]
).abs()

matches_features["combined_scoring"] = (
    matches_features["home_rolling_scored"] +
    matches_features["away_rolling_scored"]
)

matches_features["combined_conceded"] = (
    matches_features["home_rolling_conceded"] +
    matches_features["away_rolling_conceded"]
)

matches_features["total_balance"] = (
    matches_features["attack_diff"] +
    matches_features["defense_diff"]
)

matches_features["points_diff_abs"] = (
    matches_features["points_diff"]
).abs()

matches_features["position_diff_abs"] = (
    matches_features["position_diff"]
).abs()

matches_features["draw_rate_diff"] = (
    matches_features["home_draw_rate_last_10"] -
    matches_features["away_draw_rate_last_10"]
).abs()

matches_features["draw_rate_sum"] = (
    matches_features["home_draw_rate_last_10"] +
    matches_features["away_draw_rate_last_10"]
)

matches_features["rest_days_diff"] = (
    matches_features["home_rest_days"] -
    matches_features["away_rest_days"]
).abs()

matches_features["coach_tenure_diff"] = (
    matches_features["home_coach_tenure_days"] -
    matches_features["away_coach_tenure_days"]
).abs()

matches_features["home_advantage_strength"] = (
    matches_features["home_rolling_scored"] -
    matches_features["away_rolling_conceded"]
)

matches_features["away_advantage_strength"] = (
    matches_features["away_rolling_scored"] -
    matches_features["home_rolling_conceded"]
)

matches_features["strength_gap"] = (
    matches_features["home_advantage_strength"] -
    matches_features["away_advantage_strength"]
).abs()

matches_features["table_pressure_match"] = (
    matches_features["home_title_race"] +
    matches_features["away_title_race"] +
    matches_features["home_europe_race"] +
    matches_features["away_europe_race"] +
    matches_features["home_relegation_risk"] +
    matches_features["away_relegation_risk"]
)

matches_features["low_scoring_profile"] = (
    matches_features["combined_scoring"] <= 2.2
).astype(int)

matches_features["balanced_match_flag"] = (
    (
        matches_features["elo_diff"] <= 50
    ) &
    (
        matches_features["position_diff_abs"] <= 3
    ) &
    (
        matches_features["points_diff_abs"] <= 6
    )
).astype(int)

In [9]:
# ===========================================
# MULTICLASS FEATURES
# ===========================================

multiclass_features = [

    "home_elo",
    "away_elo",

    "home_rolling_scored",
    "away_rolling_scored",

    "home_rolling_conceded",
    "away_rolling_conceded",

    "home_draw_rate_last_10",
    "away_draw_rate_last_10",

    "home_position",
    "away_position",

    "points_diff",
    "position_diff",

    "home_title_race",
    "away_title_race",

    "home_europe_race",
    "away_europe_race",

    "home_relegation_risk",
    "away_relegation_risk",

    "elo_diff",
    "attack_diff",
    "defense_diff",

    "combined_scoring",
    "combined_conceded",

    "total_balance",

    "points_diff_abs",
    "position_diff_abs",

    "draw_rate_diff",
    "draw_rate_sum",

    "home_advantage_strength",
    "away_advantage_strength",

    "strength_gap",

    "table_pressure_match",

    "low_scoring_profile",

    "balanced_match_flag",

    "round_number"
]

# ===========================================
# TARGET
# ===========================================

multiclass_target = "match_result_3class"

# ===========================================
# CLEAN DATA
# ===========================================

numeric_columns = [

    "home_rest_days",
    "away_rest_days",
    "rest_days_diff",

    "home_coach_tenure_days",
    "away_coach_tenure_days",
    "coach_tenure_diff",

    "home_points",
    "away_points",

    "home_position",
    "away_position",

    "points_diff",
    "position_diff",

    "points_diff_abs",
    "position_diff_abs",

    "home_elo",
    "away_elo",

    "elo_diff",

    "home_rolling_scored",
    "away_rolling_scored",

    "home_rolling_conceded",
    "away_rolling_conceded",

    "attack_diff",
    "defense_diff",

    "combined_scoring",
    "combined_conceded",

    "total_balance",

    "home_draw_rate_last_10",
    "away_draw_rate_last_10",

    "draw_rate_diff",
    "draw_rate_sum",

    "home_advantage_strength",
    "away_advantage_strength",

    "strength_gap",

    "table_pressure_match",

    "round_number"
]

for col in numeric_columns:

    matches_features[col] = pd.to_numeric(
        matches_features[col],
        errors="coerce"
    )

multiclass_df = matches_features.dropna(
    subset=multiclass_features + ["match_result_3class"]
).copy()

# ===========================================
# TRAIN / TEST SPLIT
# ===========================================

train_df = multiclass_df[
    multiclass_df["season"] <= TRAIN_MAX_SEASON
].copy()

test_df = multiclass_df[
    multiclass_df["season"] >= TEST_MIN_SEASON
].copy()

# ===========================================
# X / Y
# ===========================================

X_train = train_df[multiclass_features]
y_train = train_df[multiclass_target]

X_test = test_df[multiclass_features]
y_test = test_df[multiclass_target]

# ===========================================
# MODEL
# ===========================================

xgb_model = XGBClassifier(

    objective="multi:softprob",

    num_class=3,

    n_estimators=400,

    max_depth=5,

    learning_rate=0.03,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    eval_metric="mlogloss"
)

# ===========================================
# TRAIN
# ===========================================

xgb_model.fit(
    X_train,
    y_train
)

# ===========================================
# PREDICTIONS
# ===========================================

predictions = xgb_model.predict(X_test)

probabilities = xgb_model.predict_proba(X_test)

# ===========================================
# REPORT
# ===========================================

print(
    classification_report(
        y_test,
        predictions
    )
)

# ===========================================
# RESULTS DF
# ===========================================

test_results = test_df.copy()

test_results["home_prob"] = probabilities[:, 0]

test_results["draw_prob"] = probabilities[:, 1]

test_results["away_prob"] = probabilities[:, 2]

test_results["prediction"] = predictions

# ===========================================
# OVERALL ACCURACY
# ===========================================

overall_accuracy = accuracy_score(
    y_test,
    predictions
)

print(
    "Overall accuracy:",
    overall_accuracy
)

              precision    recall  f1-score   support

           0       0.53      0.78      0.63      1486
           1       0.25      0.02      0.04       873
           2       0.50      0.54      0.52      1102

    accuracy                           0.51      3461
   macro avg       0.43      0.45      0.40      3461
weighted avg       0.45      0.51      0.45      3461

Overall accuracy: 0.5131464894539151


In [21]:
# ===========================================
# FUTURE MATCH HELPERS
# ===========================================

teams_df = pd.read_sql(
    """
    SELECT
        id,
        name
    FROM teams
    """,
    conn
)

team_map = dict(
    zip(
        teams_df["id"],
        teams_df["name"]
    )
)

def get_future_matches(base_df, competition_id, season=None):
    """
    Return future fixtures (status == 'NS') for a specific competition_id
    and optional season, with team names attached.
    """
    mask = (
        (base_df["status"] == "NS") &
        (base_df["competition_id"] == competition_id)
    )

    if season is not None:
        mask &= (base_df["season"] == season)

    future = base_df.loc[mask].copy()

    if future.empty:
        return future

    future["competition_name"] = COMPETITION_ID_TO_NAME.get(
        competition_id,
        str(competition_id)
    )

    future["home_team_name"] = future["home_team_id"].map(team_map)
    future["away_team_name"] = future["away_team_id"].map(team_map)

    return future

def predict_future_matches(base_df, competition_id, season=None):
    """
    Build a future fixture frame, ensure feature dtypes are numeric,
    score with the fitted XGB multiclass model, and return a sorted frame.
    """
    future = get_future_matches(base_df, competition_id, season)

    if future.empty:
        return future

    # Convert feature columns to numeric when possible.
    for col in multiclass_features:

        if col in future.columns:
    
            future[col] = pd.to_numeric(
                future[col],
                errors="coerce"
            )
    
            future[col] = future[col].fillna(
                matches_features[col].median()
            )

    future_x = future[multiclass_features]

    probabilities = xgb_model.predict_proba(future_x)

    future["home_prob"] = probabilities[:, 0]
    future["draw_prob"] = probabilities[:, 1]
    future["away_prob"] = probabilities[:, 2]

    future["top_diff"] = (
        future[["home_prob", "draw_prob", "away_prob"]]
        .apply(
            lambda row: np.sort(row.values)[-1] - np.sort(row.values)[-2],
            axis=1
        )
    )

    return future.sort_values(
        by=["top_diff", "date"],
        ascending=[True, True]
    ).reset_index(drop=True)

In [25]:
# ===========================================
# SINGLE LEAGUE FORECAST EXAMPLE
# ===========================================

FORECAST_COMPETITION_ID = 2   # 1=Premier League, 2=La Liga, 3=Serie A, 4=Bundesliga, 5=Ligue 1
FORECAST_SEASON = 2025

future_matches = predict_future_matches(
    matches_features,
    competition_id=FORECAST_COMPETITION_ID,
    season=FORECAST_SEASON
)

future_matches[
    [
        "date",
        "competition_name",
        "home_team_name",
        "away_team_name",
        "home_prob",
        "draw_prob",
        "away_prob",
        "top_diff"
    ]
].head(50)

,date,competition_name,home_team_name,away_team_name,home_prob,draw_prob,away_prob,top_diff
0,2026-05-23 19:00:00.000000,La Liga,Celta Vigo,Sevilla,0.422578,0.352594,0.224828,0.069983
1,2026-05-23 19:00:00.000000,La Liga,Alaves,Rayo Vallecano,0.410440,0.288505,0.301055,0.109385
2,2026-05-23 19:00:00.000000,La Liga,Mallorca,Oviedo,0.429565,0.298176,0.272259,0.131389
3,2026-05-23 19:00:00.000000,La Liga,Girona,Elche,0.442315,0.284894,0.272791,0.157421
4,2026-05-23 19:00:00.000000,La Liga,Espanyol,Real Sociedad,0.485646,0.208170,0.306185,0.179461
5,2026-05-23 19:00:00.000000,La Liga,Getafe,Osasuna,0.461531,0.275091,0.263377,0.186440
6,2026-05-24 19:00:00.000000,La Liga,Villarreal,Atletico Madrid,0.525520,0.231903,0.242577,0.282943
7,2026-05-23 19:00:00.000000,La Liga,Real Betis,Levante,0.608492,0.212753,0.178755,0.395739
8,2026-05-23 19:00:00.000000,La Liga,Valencia,Barcelona,0.199366,0.183038,0.617596,0.418231
9,2026-05-23 19:00:00.000000,La Liga,Real Madrid,Athletic Club,0.757990,0.136302,0.105708,0.621688


In [24]:
# ===========================================
# ALL EUROPEAN LEAGUES FORECAST
# ===========================================

FORECAST_SEASON = 2025

all_future_predictions = []

for competition_id in EUROPEAN_LEAGUES:
    league_future = predict_future_matches(
        matches_features,
        competition_id=competition_id,
        season=FORECAST_SEASON
    )

    if not league_future.empty:
        all_future_predictions.append(league_future)

if all_future_predictions:
    all_future_predictions = pd.concat(
        all_future_predictions,
        ignore_index=True
    )

    all_future_predictions[
        [
            "date",
            "competition_name",
            "home_team_name",
            "away_team_name",
            "home_prob",
            "draw_prob",
            "away_prob",
            "top_diff"
        ]
    ].sort_values(
        by=["competition_name", "top_diff", "date"],
        ascending=[True, True, True]
    ).head(100)
else:
    all_future_predictions = pd.DataFrame()
    print("No future matches found for the selected season/leagues.")